## 1. Convert joining_date to datetime and count how many rows failed conversion.

In [1]:
import pandas as pd
df = pd.read_csv('Employee_Dataset.csv')
df

,employee_id,department,designation,age,salary,joining_date,last_promotion_date,experience_years,performance_rating,is_active
0,EMP1000,IT,Senior Analyst,60,85000,invalid,01/04/2021,5,3,NaN
1,emp_1,IT,Analyst,NaN,55000,10/06/2020,invalid,5,3,True
2,EMP1002,Sales,NaN,28,85000,NaN,2022-03-01,3,4,NaN
3,EMP1003,it,Analyst,45,35000,invalid,01/04/2021,1,3,NaN
4,NaN,IT,mgr,150,85000,NaN,invalid,12,2,True
...,...,...,...,...,...,...,...,...,...,...
995,EMP1995,Finance,NaN,60,55000,2019-05-10,2022-03-01,1,4,yes
996,EMP1996,IT,Analyst,35,250000,invalid,invalid,-2,5,NaN
997,EMP1997,Sales,Senior Analyst,150,250000,2021/07/15,invalid,3,excellent,yes
998,emp_998,Finance,NaN,60,85000,2019-05-10,NaN,12,NaN,True


In [2]:
df['joining_date_clean'] = df['joining_date'].astype(str).str.replace('/', '-', regex=False)
df['joining_date_clean'] = pd.to_datetime(df['joining_date_clean'],dayfirst=True,errors='coerce')
failed_rows = df['joining_date_clean'].isna().sum()
failed_rows



C:\Users\SATHWIKA\AppData\Local\Temp\ipykernel_7492\2839812444.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['joining_date_clean'] = pd.to_datetime(df['joining_date_clean'],dayfirst=True,errors='coerce')


np.int64(377)

## 2. Clean employee_id and identify how many duplicate employees exist after standardization.

In [3]:

df['employee_id_clean'] = (df['employee_id'].astype(str).str.strip().str.upper())

duplicate_count = df['employee_id_clean'].duplicated().sum()
duplicate_count



np.int64(503)

## 3. Standardize department and calculate the average salary per department excluding invalid salaries.

In [4]:
df['department'] = df['department'].str.strip().str.title()
df['salary'] = pd.to_numeric(df['salary'], errors='coerce')
df.groupby('department')['salary'].mean()


department
Finance    113253.012048
Hr         114800.000000
It         107944.444444
Sales      119000.000000
Name: salary, dtype: float64

## 4. Convert age to numeric and find employees with valid salary but invalid age.

In [5]:
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df[df['salary'].notna() & df['age'].isna()]


,employee_id,department,designation,age,salary,joining_date,last_promotion_date,experience_years,performance_rating,is_active,joining_date_clean,employee_id_clean
1,emp_1,It,Analyst,NaN,55000.0,10/06/2020,invalid,5,3,True,2020-06-10,EMP_1
30,EMP1030,Hr,Manager,NaN,85000.0,2019-05-10,invalid,3,3,True,2019-05-10,EMP1030
31,NaN,Hr,Analyst,NaN,55000.0,10/06/2020,2022-03-01,1,4,NaN,2020-06-10,NAN
38,emp_38,It,Manager,NaN,35000.0,2021/07/15,2022-03-01,1,poor,True,2021-07-15,EMP_38
45,NaN,Finance,NaN,NaN,120000.0,NaN,01/04/2021,1,1,yes,NaT,NAN
...,...,...,...,...,...,...,...,...,...,...,...,...
959,NaN,Hr,NaN,NaN,120000.0,10/06/2020,invalid,8,3,NaN,2020-06-10,NAN
974,emp_974,NaN,mgr,NaN,85000.0,2021/07/15,01/04/2021,3,4,False,2021-07-15,EMP_974
976,EMP1976,Hr,Analyst,NaN,250000.0,2019-05-10,NaN,1,poor,False,2019-05-10,EMP1976
982,NaN,Hr,NaN,NaN,55000.0,10/06/2020,NaN,5,NaN,NaN,2020-06-10,NAN


## 5. Clean salary and detect outliers using the IQR method.

In [6]:
Q1 = df['salary'].quantile(0.25)
Q3 = df['salary'].quantile(0.75)
IQR = Q3 - Q1
df[(df['salary'] < Q1 - 1.5*IQR) | (df['salary'] > Q3 + 1.5*IQR)]


,employee_id,department,designation,age,salary,joining_date,last_promotion_date,experience_years,performance_rating,is_active,joining_date_clean,employee_id_clean
11,NaN,Sales,mgr,22.0,250000.0,2021/07/15,NaN,NaN,5,yes,2021-07-15,NAN
16,NaN,It,NaN,60.0,250000.0,NaN,invalid,12,poor,no,NaT,NAN
25,NaN,Sales,NaN,28.0,250000.0,invalid,NaN,8,NaN,True,NaT,NAN
34,NaN,Sales,Analyst,45.0,250000.0,NaN,2022-03-01,-2,5,False,NaT,NAN
41,NaN,NaN,ANALYST,60.0,250000.0,2021/07/15,NaN,3,4,yes,2021-07-15,NAN
...,...,...,...,...,...,...,...,...,...,...,...,...
955,NaN,It,ANALYST,60.0,250000.0,2019-05-10,01/04/2021,NaN,3,no,2019-05-10,NAN
969,EMP1969,It,NaN,150.0,250000.0,2019-05-10,2022-03-01,NaN,NaN,NaN,2019-05-10,EMP1969
976,EMP1976,Hr,Analyst,NaN,250000.0,2019-05-10,NaN,1,poor,False,2019-05-10,EMP1976
996,EMP1996,It,Analyst,35.0,250000.0,invalid,invalid,-2,5,NaN,NaT,EMP1996


## 6. Convert performance_rating into numeric and calculate the median rating per designation.

In [7]:
df['performance_rating'] = pd.to_numeric(df['performance_rating'], errors='coerce')
df.groupby('designation')['performance_rating'].median()


designation
ANALYST           3.0
Analyst           3.0
Manager           3.0
Senior Analyst    3.0
mgr               3.0
Name: performance_rating, dtype: float64

## 7. Identify employees whose last_promotion_date is earlier than their joining_date.

In [8]:
df['joining_date'] = pd.to_datetime(df['joining_date'], errors='coerce')
df['last_promotion_date'] = pd.to_datetime(df['last_promotion_date'], errors='coerce')

df[df['last_promotion_date'] < df['joining_date']]



C:\Users\SATHWIKA\AppData\Local\Temp\ipykernel_7492\1990220903.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['joining_date'] = pd.to_datetime(df['joining_date'], errors='coerce')


,employee_id,department,designation,age,salary,joining_date,last_promotion_date,experience_years,performance_rating,is_active,joining_date_clean,employee_id_clean
24,EMP1024,Finance,mgr,35.0,85000.0,2021-07-15,2021-01-04,8,3.0,True,2021-07-15,EMP1024
40,emp_40,NaN,NaN,60.0,85000.0,2021-07-15,2021-01-04,12,5.0,True,2021-07-15,EMP_40
44,emp_44,Sales,mgr,45.0,NaN,2021-07-15,2021-01-04,8,2.0,True,2021-07-15,EMP_44
47,NaN,Finance,Analyst,NaN,250000.0,2021-07-15,2021-01-04,3,NaN,True,2021-07-15,NAN
71,emp_71,Sales,Manager,60.0,250000.0,2021-07-15,2021-01-04,12,NaN,False,2021-07-15,EMP_71
82,emp_82,Sales,Senior Analyst,150.0,NaN,2021-07-15,2021-01-04,-2,1.0,yes,2021-07-15,EMP_82
85,NaN,Sales,NaN,60.0,55000.0,2021-07-15,2021-01-04,NaN,2.0,yes,2021-07-15,NAN
124,NaN,Sales,Analyst,NaN,250000.0,2021-07-15,2021-01-04,1,5.0,no,2021-07-15,NAN
138,emp_138,Finance,mgr,45.0,250000.0,2021-07-15,2021-01-04,NaN,3.0,False,2021-07-15,EMP_138
169,emp_169,It,mgr,60.0,120000.0,2021-07-15,2021-01-04,NaN,2.0,NaN,2021-07-15,EMP_169


## 8. Clean experience_years and find mismatches where experience exceeds employee age.

In [9]:
df['experience_years'] = pd.to_numeric(df['experience_years'], errors='coerce')
df[df['experience_years'] > df['age']]


,employee_id,department,designation,age,salary,joining_date,last_promotion_date,experience_years,performance_rating,is_active,joining_date_clean,employee_id_clean
10,emp_10,Finance,Analyst,-5.0,35000.0,NaT,NaT,8.0,1.0,no,NaT,EMP_10
17,emp_17,Sales,Senior Analyst,-5.0,NaN,NaT,2021-01-04,3.0,4.0,False,NaT,EMP_17
23,emp_23,Finance,NaN,-5.0,85000.0,NaT,NaT,-2.0,5.0,NaN,NaT,EMP_23
43,NaN,It,Manager,-5.0,120000.0,NaT,NaT,1.0,NaN,NaN,NaT,NAN
49,NaN,It,mgr,-5.0,55000.0,2021-07-15,NaT,5.0,3.0,False,2021-07-15,NAN
...,...,...,...,...,...,...,...,...,...,...,...,...
965,NaN,Sales,mgr,-5.0,NaN,2020-10-06,NaT,12.0,NaN,no,2020-06-10,NAN
981,NaN,Sales,mgr,-5.0,85000.0,NaT,NaT,-2.0,NaN,yes,NaT,NAN
985,emp_985,Sales,mgr,-5.0,NaN,NaT,NaT,-2.0,1.0,NaN,NaT,EMP_985
990,emp_990,NaN,Senior Analyst,-5.0,85000.0,2019-05-10,2021-01-04,12.0,5.0,yes,2019-05-10,EMP_990


## 9. Standardize designation and count how many active employees are in each designation.

In [10]:
df['is_active'] = df['is_active'].astype(str).str.strip().str.lower()
df[df['is_active'] == 'yes'].groupby('designation').size()


designation
ANALYST           23
Analyst           35
Manager           34
Senior Analyst    58
mgr               39
dtype: int64

## 10.Convert is_active to boolean and find inactive employees with recent promotions.

In [11]:
df.loc[(df['is_active'] == False) &(df['last_promotion_date'] >= pd.Timestamp.now() - pd.DateOffset(years=5)),['employee_id', 'is_active', 'last_promotion_date']
]


,employee_id,is_active,last_promotion_date


## 11.Calculate employee tenure in years and find those with tenure above the 90th percentile.

In [12]:
df['joining_date'] = pd.to_datetime(df['joining_date'], errors='coerce')

df['tenure_years'] = (pd.Timestamp.now() - df['joining_date']).dt.days / 365

tenure_90 = df['tenure_years'].quantile(0.9)

df[df['tenure_years'] >= tenure_90][['employee_id', 'tenure_years']]



,employee_id,tenure_years
5,NaN,6.764384
14,NaN,6.764384
26,NaN,6.764384
29,NaN,6.764384
30,EMP1030,6.764384
...,...,...
980,EMP1980,6.764384
990,emp_990,6.764384
994,EMP1994,6.764384
995,EMP1995,6.764384


## 12.Identify departments where more than 25% of salary values are missing or invalid.

In [13]:
df['salary'] = pd.to_numeric(df['salary'], errors='coerce')

dept_salary_missing = df.groupby('department')['salary'].apply(
    lambda x: x.isna().mean()
)

dept_salary_missing[dept_salary_missing > 0.25]


department
Finance    0.366412
Hr         0.321267
It         0.340659
Sales      0.398340
Name: salary, dtype: float64

## 13.Create a flag for employees with high performance (≥4) but below-median salary

In [14]:
df['performance_rating'] = pd.to_numeric(df['performance_rating'], errors='coerce')

median_salary = df['salary'].median()

df[(df['performance_rating'] >= 4) & (df['salary'] < median_salary)][
    ['employee_id', 'performance_rating', 'salary']
]


,employee_id,performance_rating,salary
21,NaN,4.0,55000.0
29,NaN,4.0,55000.0
31,NaN,4.0,55000.0
66,NaN,4.0,35000.0
92,NaN,5.0,35000.0
118,NaN,4.0,35000.0
145,EMP1145,4.0,35000.0
149,EMP1149,5.0,35000.0
189,NaN,4.0,55000.0
208,emp_208,5.0,55000.0


## 14.Detect employees with no promotion date but more than 5 years of experience.

In [15]:
df['experience_years'] = pd.to_numeric(df['experience_years'], errors='coerce')

df[df['last_promotion_date'].isna() & (df['experience_years'] > 5)][
    ['employee_id', 'experience_years']
]


,employee_id,experience_years
4,NaN,12.0
5,NaN,8.0
6,NaN,12.0
10,emp_10,8.0
12,emp_12,8.0
...,...,...
977,emp_977,8.0
978,EMP1978,8.0
993,EMP1993,12.0
994,EMP1994,12.0


## 15.1 After cleaning dates, find employees whose promotion gap (promotion date − joining date) is less than 1 year.

In [16]:
promo_gap_days = (df['last_promotion_date'] - df['joining_date']).dt.days

df[promo_gap_days < 365][['employee_id', 'joining_date', 'last_promotion_date']]


,employee_id,joining_date,last_promotion_date
9,EMP1009,2020-10-06,2021-01-04
24,EMP1024,2021-07-15,2021-01-04
40,emp_40,2021-07-15,2021-01-04
44,emp_44,2021-07-15,2021-01-04
47,NaN,2021-07-15,2021-01-04
...,...,...,...
943,EMP1943,2020-10-06,2021-01-04
945,NaN,2020-10-06,2021-01-04
966,NaN,2020-10-06,2021-01-04
974,emp_974,2021-07-15,2021-01-04


## 15.2 Identify employees whose salary is above the department average but performance rating is below the department median.

In [17]:
dept_avg_salary = df.groupby('department')['salary'].transform('mean')
dept_med_perf = df.groupby('department')['performance_rating'].transform('median')

df[(df['salary'] > dept_avg_salary) &
   (df['performance_rating'] < dept_med_perf)]


,employee_id,department,designation,age,salary,joining_date,last_promotion_date,experience_years,performance_rating,is_active,joining_date_clean,employee_id_clean,tenure_years
45,NaN,Finance,NaN,NaN,120000.0,NaT,2021-01-04,1.0,1.0,yes,NaT,NAN,NaN
115,emp_115,Sales,ANALYST,28.0,250000.0,2021-07-15,NaT,-2.0,2.0,no,2021-07-15,EMP_115,4.580822
127,NaN,Finance,NaN,150.0,120000.0,NaT,NaT,3.0,1.0,true,NaT,NAN,NaN
157,NaN,Finance,ANALYST,22.0,250000.0,2019-05-10,NaT,3.0,1.0,false,2019-05-10,NAN,6.764384
169,emp_169,It,mgr,60.0,120000.0,2021-07-15,2021-01-04,NaN,2.0,nan,2021-07-15,EMP_169,4.580822
173,emp_173,Finance,Manager,45.0,120000.0,2020-10-06,NaT,8.0,1.0,false,2020-06-10,EMP_173,5.353425
217,NaN,It,NaN,28.0,250000.0,2021-07-15,NaT,NaN,2.0,yes,2021-07-15,NAN,4.580822
223,NaN,Hr,Manager,NaN,120000.0,2019-05-10,NaT,-2.0,1.0,true,2019-05-10,NAN,6.764384
270,NaN,It,ANALYST,NaN,120000.0,2020-10-06,NaT,-2.0,2.0,nan,2020-06-10,NAN,5.353425
273,NaN,Finance,ANALYST,NaN,120000.0,2020-10-06,NaT,12.0,2.0,yes,2020-06-10,NAN,5.353425


## 15.3 Create a column that categorizes employees into age bands (Young, Mid, Senior) and count employees per band per department.

In [18]:
df['age_band'] = pd.cut(
    df['age'],
    bins=[0, 30, 50, 100],
    labels=['Young', 'Mid', 'Senior']
)
df.groupby(['department', 'age_band']).size()


C:\Users\SATHWIKA\AppData\Local\Temp\ipykernel_7492\1532321958.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['department', 'age_band']).size()


department  age_band
Finance     Young       33
            Mid         31
            Senior      13
Hr          Young       54
            Mid         52
            Senior      22
It          Young       53
            Mid         78
            Senior      37
Sales       Young       48
            Mid         57
            Senior      25
dtype: int64

## 15.4 Find departments where the average experience is higher than the company-wide average experience.

In [19]:
company_avg_exp = df['experience_years'].mean()
dept_avg_exp = df.groupby('department')['experience_years'].mean()
dept_avg_exp[dept_avg_exp > company_avg_exp]


department
Finance    4.826923
Sales      4.709677
Name: experience_years, dtype: float64

## 15.5 Detect employees marked as inactive but having a valid promotion date within the last 2 years.

In [20]:
df[(df['is_active'] == 'no') & (df['last_promotion_date'] >= pd.Timestamp.now() - pd.DateOffset(years=2))]


,employee_id,department,designation,age,salary,joining_date,last_promotion_date,experience_years,performance_rating,is_active,joining_date_clean,employee_id_clean,tenure_years,age_band


## 15.6 Identify designations where more than 20% of employees have missing or invalid experience values.

In [21]:
invalid_exp_ratio = df.groupby('designation')['experience_years'] \
    .apply(lambda x: x.isna().mean())
invalid_exp_ratio[invalid_exp_ratio > 0.20]


designation
ANALYST           0.224638
Senior Analyst    0.202128
mgr               0.264045
Name: experience_years, dtype: float64

## 15.7 Calculate the ratio of salary to experience years and detect extreme values using the 95th percentile.

In [22]:
df['salary_exp_ratio'] = df['salary'] / df['experience_years']
threshold_95 = df['salary_exp_ratio'].quantile(0.95)
df[df['salary_exp_ratio'] > threshold_95]


,employee_id,department,designation,age,salary,joining_date,last_promotion_date,experience_years,performance_rating,is_active,joining_date_clean,employee_id_clean,tenure_years,age_band,salary_exp_ratio
124,NaN,Sales,Analyst,NaN,250000.0,2021-07-15,2021-01-04,1.0,5.0,no,2021-07-15,NAN,4.580822,NaN,250000.0
133,NaN,It,NaN,-5.0,250000.0,2021-07-15,NaT,1.0,5.0,true,2021-07-15,NAN,4.580822,NaN,250000.0
176,NaN,It,Analyst,22.0,250000.0,NaT,NaT,1.0,NaN,false,NaT,NAN,NaN,Young,250000.0
224,NaN,Finance,Manager,60.0,250000.0,2021-07-15,NaT,1.0,3.0,nan,2021-07-15,NAN,4.580822,Senior,250000.0
262,NaN,It,Manager,-5.0,250000.0,NaT,NaT,1.0,5.0,nan,NaT,NAN,NaN,NaN,250000.0
399,NaN,Finance,ANALYST,60.0,250000.0,NaT,NaT,1.0,3.0,true,NaT,NAN,NaN,Senior,250000.0
401,EMP1401,Finance,Analyst,22.0,250000.0,NaT,NaT,1.0,1.0,nan,NaT,EMP1401,NaN,Young,250000.0
470,EMP1470,Hr,mgr,35.0,250000.0,2021-07-15,NaT,1.0,NaN,true,2021-07-15,EMP1470,4.580822,Mid,250000.0
525,emp_525,Sales,Manager,-5.0,250000.0,2020-10-06,NaT,1.0,1.0,false,2020-06-10,EMP_525,5.353425,NaN,250000.0
529,emp_529,NaN,NaN,28.0,250000.0,2020-10-06,NaT,1.0,5.0,no,2020-06-10,EMP_529,5.353425,Young,250000.0


## 15.8 Find employees whose age is valid but inconsistent with experience (experience > age − 18).

In [23]:
df[df['age'].notna() & (df['experience_years'] > (df['age'] - 18))]


,employee_id,department,designation,age,salary,joining_date,last_promotion_date,experience_years,performance_rating,is_active,joining_date_clean,employee_id_clean,tenure_years,age_band,salary_exp_ratio
10,emp_10,Finance,Analyst,-5.0,35000.0,NaT,NaT,8.0,1.0,no,NaT,EMP_10,NaN,NaN,4375.000000
17,emp_17,Sales,Senior Analyst,-5.0,NaN,NaT,2021-01-04,3.0,4.0,false,NaT,EMP_17,NaN,NaN,NaN
23,emp_23,Finance,NaN,-5.0,85000.0,NaT,NaT,-2.0,5.0,nan,NaT,EMP_23,NaN,NaN,-42500.000000
43,NaN,It,Manager,-5.0,120000.0,NaT,NaT,1.0,NaN,nan,NaT,NAN,NaN,NaN,120000.000000
49,NaN,It,mgr,-5.0,55000.0,2021-07-15,NaT,5.0,3.0,false,2021-07-15,NAN,4.580822,NaN,11000.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
968,emp_968,It,Manager,22.0,120000.0,2021-07-15,NaT,5.0,NaN,true,2021-07-15,EMP_968,4.580822,Young,24000.000000
981,NaN,Sales,mgr,-5.0,85000.0,NaT,NaT,-2.0,NaN,yes,NaT,NAN,NaN,NaN,-42500.000000
985,emp_985,Sales,mgr,-5.0,NaN,NaT,NaT,-2.0,1.0,nan,NaT,EMP_985,NaN,NaN,NaN
990,emp_990,NaN,Senior Analyst,-5.0,85000.0,2019-05-10,2021-01-04,12.0,5.0,yes,2019-05-10,EMP_990,6.764384,NaN,7083.333333


## 15.9 Compute department-wise attrition rate assuming inactive employees represent attrition.

In [24]:
attrition_rate = df.groupby('department')['is_active'] \
    .apply(lambda x: (x == 'no').mean())

attrition_rate


department
Finance    0.167939
Hr         0.153846
It         0.164835
Sales      0.232365
Name: is_active, dtype: float64

## 15.10 Create a final quality score per row based on number of valid fields and flag rows below a quality threshold.

In [25]:
df['quality_score'] = (
    df['age'].notna().astype(int) +
    df['salary'].notna().astype(int) +
    df['experience_years'].notna().astype(int) +
    df['joining_date'].notna().astype(int)
)

df[df['quality_score'] < 3][['employee_id', 'quality_score']]


,employee_id,quality_score
5,NaN,2
12,emp_12,2
17,emp_17,2
18,emp_18,0
19,NaN,2
...,...,...
973,NaN,2
983,EMP1983,2
984,NaN,2
985,emp_985,2
